In [1]:
%pip install -q pixeltable-pgserver "psycopg[binary]" sqlalchemy pandas pyarrow ucimlrepo

import pandas as pd
import pixeltable_pgserver as pgserver
from sqlalchemy import create_engine, text

from pathlib import Path

pg = pgserver.get_server(
    '/tmp/hw3_pg',
    cleanup_mode='stop'
)

engine = create_engine(
    pg.get_uri(),
    future=True
)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print("PostgreSQL:", version)

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:467: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


PostgreSQL: PostgreSQL 16.15 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


In [2]:
from pathlib import Path
from urllib.request import urlretrieve
import pandas as pd

parquet_path = Path("hw3_taxi_month.parquet")
csv_path = Path("hw3_taxi_100k.csv")

url = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_2024-01.parquet"
)

columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount",
]

# Завантажуємо Parquet, якщо його ще немає
if not parquet_path.exists():
    urlretrieve(url, parquet_path)

# Читаємо перші 100 000 рядків
df = pd.read_parquet(
    parquet_path,
    columns=columns
).head(100_000)

# Зберігаємо CSV
df.to_csv(csv_path, index=False)

print("DataFrame shape:", df.shape)
print("Parquet exists:", parquet_path.exists())
print("CSV exists:", csv_path.exists())
print("CSV size (MB):", round(csv_path.stat().st_size / 1024**2, 2))

DataFrame shape: (100000, 11)
Parquet exists: True
CSV exists: True
CSV size (MB): 7.16


In [3]:
from sqlalchemy import text

create_staging_sql = """
CREATE TABLE IF NOT EXISTS hw3_taxi_staging (
    "VendorID" TEXT,
    tpep_pickup_datetime TEXT,
    tpep_dropoff_datetime TEXT,
    passenger_count TEXT,
    trip_distance TEXT,
    "PULocationID" TEXT,
    "DOLocationID" TEXT,
    payment_type TEXT,
    fare_amount TEXT,
    tip_amount TEXT,
    total_amount TEXT
);
"""

with engine.begin() as conn:
    conn.execute(text(create_staging_sql))

print("Staging table created")

copy_sql = """
COPY hw3_taxi_staging (
    "VendorID",
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    passenger_count,
    trip_distance,
    "PULocationID",
    "DOLocationID",
    payment_type,
    fare_amount,
    tip_amount,
    total_amount
)
FROM STDIN
WITH (FORMAT CSV, HEADER TRUE)
"""

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE hw3_taxi_staging"))

    raw_conn = conn.connection.driver_connection

    with raw_conn.cursor() as cursor:
        with cursor.copy(copy_sql) as copy:
            with open(csv_path, "rb") as file:
                while chunk := file.read(1024 * 1024):
                    copy.write(chunk)

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM hw3_taxi_staging")
    ).scalar_one()

print("Staging rows:", count)

Staging table created
Staging rows: 100000


In [4]:
create_clean_sql = """
CREATE TABLE IF NOT EXISTS hw3_taxi_clean (
    trip_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    vendor_id INTEGER,
    pickup_datetime TIMESTAMPTZ NOT NULL,
    dropoff_datetime TIMESTAMPTZ NOT NULL,
    passenger_count INTEGER,
    trip_distance NUMERIC(10, 2),
    pu_location_id INTEGER,
    do_location_id INTEGER,
    payment_type INTEGER,
    fare_amount NUMERIC(12, 2),
    tip_amount NUMERIC(12, 2),
    total_amount NUMERIC(12, 2),

    CONSTRAINT chk_trip_distance
        CHECK (trip_distance >= 0),

    CONSTRAINT chk_trip_duration
        CHECK (dropoff_datetime >= pickup_datetime)
);
"""

with engine.begin() as conn:
    conn.execute(text(create_clean_sql))

print("Clean table created")

insert_clean_sql = """
INSERT INTO hw3_taxi_clean (
    vendor_id,
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    pu_location_id,
    do_location_id,
    payment_type,
    fare_amount,
    tip_amount,
    total_amount
)
SELECT
    NULLIF("VendorID", '')::INTEGER,
    NULLIF(tpep_pickup_datetime, '')::TIMESTAMP
        AT TIME ZONE 'America/New_York',
    NULLIF(tpep_dropoff_datetime, '')::TIMESTAMP
        AT TIME ZONE 'America/New_York',
    NULLIF(passenger_count, '')::NUMERIC::INTEGER,
    NULLIF(trip_distance, '')::NUMERIC(10, 2),
    NULLIF("PULocationID", '')::INTEGER,
    NULLIF("DOLocationID", '')::INTEGER,
    NULLIF(payment_type, '')::INTEGER,
    NULLIF(fare_amount, '')::NUMERIC(12, 2),
    NULLIF(tip_amount, '')::NUMERIC(12, 2),
    NULLIF(total_amount, '')::NUMERIC(12, 2)
FROM hw3_taxi_staging
WHERE
    NULLIF(tpep_pickup_datetime, '') IS NOT NULL
    AND NULLIF(tpep_dropoff_datetime, '') IS NOT NULL
    AND NULLIF(trip_distance, '')::NUMERIC >= 0
    AND NULLIF(tpep_dropoff_datetime, '')::TIMESTAMP
        >= NULLIF(tpep_pickup_datetime, '')::TIMESTAMP
"""

with engine.begin() as conn:
    conn.execute(
        text("TRUNCATE TABLE hw3_taxi_clean RESTART IDENTITY")
    )
    conn.execute(text(insert_clean_sql))

with engine.connect() as conn:
    staging_count = conn.execute(
        text("SELECT COUNT(*) FROM hw3_taxi_staging")
    ).scalar_one()

    clean_count = conn.execute(
        text("SELECT COUNT(*) FROM hw3_taxi_clean")
    ).scalar_one()

print("Staging:", staging_count)
print("Clean:", clean_count)
print("Rejected:", staging_count - clean_count)

Clean table created
Staging: 100000
Clean: 99999
Rejected: 1


In [5]:
dq_sql = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE vendor_id IS NULL
    ) AS null_vendor,

    COUNT(*) FILTER (
        WHERE passenger_count IS NULL
    ) AS null_passengers,

    COUNT(*) FILTER (
        WHERE trip_distance IS NULL
    ) AS null_distance,

    COUNT(*) FILTER (
        WHERE total_amount IS NULL
    ) AS null_total,

    COUNT(*) FILTER (
        WHERE passenger_count = 0
    ) AS zero_passengers,

    COUNT(*) FILTER (
        WHERE trip_distance = 0
    ) AS zero_distance,

    COUNT(*) FILTER (
        WHERE total_amount < 0
    ) AS negative_total,

    COUNT(*) FILTER (
        WHERE trip_distance > 100
    ) AS distance_over_100

FROM hw3_taxi_clean
"""

duplicates_sql = """
SELECT COUNT(*) AS duplicate_groups
FROM (
    SELECT
        vendor_id,
        pickup_datetime,
        dropoff_datetime,
        passenger_count,
        trip_distance,
        pu_location_id,
        do_location_id,
        payment_type,
        fare_amount,
        tip_amount,
        total_amount
    FROM hw3_taxi_clean
    GROUP BY
        vendor_id,
        pickup_datetime,
        dropoff_datetime,
        passenger_count,
        trip_distance,
        pu_location_id,
        do_location_id,
        payment_type,
        fare_amount,
        tip_amount,
        total_amount
    HAVING COUNT(*) > 1
) AS duplicates
"""

with engine.connect() as conn:
    dq = conn.execute(text(dq_sql)).mappings().one()
    duplicates = conn.execute(
        text(duplicates_sql)
    ).scalar_one()

    print("DATA QUALITY")
    for key, value in dq.items():
        print(f"{key}: {value}")

    print("duplicate_groups:", duplicates)

DATA QUALITY
total_rows: 99999
null_vendor: 0
null_passengers: 0
null_distance: 0
null_total: 0
zero_passengers: 1047
zero_distance: 1811
negative_total: 1586
distance_over_100: 4
duplicate_groups: 0


In [6]:
stats_sql = """
SELECT
    COUNT(*) AS total_trips,

    ROUND(AVG(trip_distance), 2) AS avg_distance,
    MIN(trip_distance) AS min_distance,
    MAX(trip_distance) AS max_distance,
    PERCENTILE_CONT(0.5)
        WITHIN GROUP (ORDER BY trip_distance)
        AS median_distance,
    ROUND(STDDEV(trip_distance), 2) AS std_distance,

    ROUND(AVG(total_amount), 2) AS avg_amount,
    MIN(total_amount) AS min_amount,
    MAX(total_amount) AS max_amount,
    PERCENTILE_CONT(0.5)
        WITHIN GROUP (ORDER BY total_amount)
        AS median_amount,
    ROUND(STDDEV(total_amount), 2) AS std_amount

FROM hw3_taxi_clean
"""

distribution_queries = {
    "Vendor": """
        SELECT vendor_id, COUNT(*) AS trips
        FROM hw3_taxi_clean
        GROUP BY vendor_id
        ORDER BY trips DESC
    """,
    "Payment type": """
        SELECT payment_type, COUNT(*) AS trips
        FROM hw3_taxi_clean
        GROUP BY payment_type
        ORDER BY trips DESC
    """,
    "Passenger count": """
        SELECT passenger_count, COUNT(*) AS trips
        FROM hw3_taxi_clean
        GROUP BY passenger_count
        ORDER BY trips DESC
    """
}

with engine.connect() as conn:
    print("NUMERIC STATISTICS")
    print(dict(conn.execute(text(stats_sql)).mappings().one()))

    for name, sql in distribution_queries.items():
        print(f"\n{name}")
        for row in conn.execute(text(sql)).mappings():
            print(dict(row))

NUMERIC STATISTICS
{'total_trips': 99999, 'avg_distance': Decimal('4.20'), 'min_distance': Decimal('0.00'), 'max_distance': Decimal('971.80'), 'median_distance': 2.0, 'std_distance': Decimal('6.92'), 'avg_amount': Decimal('30.04'), 'min_amount': Decimal('-426.54'), 'max_amount': Decimal('1617.50'), 'median_amount': 20.3, 'std_amount': Decimal('28.20')}

Vendor
{'vendor_id': 2, 'trips': 77128}
{'vendor_id': 1, 'trips': 22871}

Payment type
{'payment_type': 1, 'trips': 75747}
{'payment_type': 2, 'trips': 21277}
{'payment_type': 4, 'trips': 2129}
{'payment_type': 3, 'trips': 846}

Passenger count
{'passenger_count': 1, 'trips': 68694}
{'passenger_count': 2, 'trips': 18708}
{'passenger_count': 3, 'trips': 5364}
{'passenger_count': 4, 'trips': 4097}
{'passenger_count': 5, 'trips': 1308}
{'passenger_count': 0, 'trips': 1047}
{'passenger_count': 6, 'trips': 775}
{'passenger_count': 8, 'trips': 6}


In [7]:
business_queries = {
    "Q1. Найзавантаженіші години": """
        SELECT
            EXTRACT(HOUR FROM pickup_datetime AT TIME ZONE 'America/New_York')::INT AS hour,
            COUNT(*) AS trips
        FROM hw3_taxi_clean
        GROUP BY hour
        ORDER BY trips DESC
        LIMIT 5
    """,

    "Q2. Середня вартість за перевізником": """
        SELECT
            vendor_id,
            COUNT(*) AS trips,
            ROUND(AVG(total_amount), 2) AS avg_amount
        FROM hw3_taxi_clean
        GROUP BY vendor_id
        ORDER BY avg_amount DESC
    """,

    "Q3. Чайові за способом оплати": """
        SELECT
            payment_type,
            COUNT(*) AS trips,
            ROUND(AVG(tip_amount), 2) AS avg_tip,
            ROUND(AVG(total_amount), 2) AS avg_total
        FROM hw3_taxi_clean
        GROUP BY payment_type
        ORDER BY avg_tip DESC
    """,
    "Q4. Найпопулярніші зони посадки": """
        SELECT
            pu_location_id,
            COUNT(*) AS trips
        FROM hw3_taxi_clean
        GROUP BY pu_location_id
        ORDER BY trips DESC
        LIMIT 5
    """,

    "Q5. Середня дистанція за кількістю пасажирів": """
        SELECT
            passenger_count,
            COUNT(*) AS trips,
            ROUND(AVG(trip_distance), 2) AS avg_distance
        FROM hw3_taxi_clean
        GROUP BY passenger_count
        ORDER BY passenger_count
    """,

    "Q6. Найдорожчі поїздки": """
        SELECT
            trip_id,
            trip_distance,
            fare_amount,
            total_amount
        FROM hw3_taxi_clean
        ORDER BY total_amount DESC
        LIMIT 5
    """,

    "Q7. Середня тривалість поїздки": """
        SELECT
            ROUND(
                AVG(
                    EXTRACT(EPOCH FROM (
                        dropoff_datetime - pickup_datetime
                    )) / 60
                )::NUMERIC,
                2
            ) AS avg_duration_minutes,
            COUNT(*) AS trips
        FROM hw3_taxi_clean
    """,
     "Q8. Кількість поїздок за датами": """
        SELECT
            (pickup_datetime AT TIME ZONE 'America/New_York')::DATE AS trip_date,
            COUNT(*) AS trips,
            ROUND(AVG(total_amount), 2) AS avg_amount
        FROM hw3_taxi_clean
        GROUP BY trip_date
        ORDER BY trip_date
    """,

    "Q9. Найпопулярніші маршрути": """
        SELECT
            pu_location_id,
            do_location_id,
            COUNT(*) AS trips,
            ROUND(AVG(trip_distance), 2) AS avg_distance
        FROM hw3_taxi_clean
        GROUP BY pu_location_id, do_location_id
        ORDER BY trips DESC
        LIMIT 5
    """,

    "Q10. Частка чайових від вартості поїздки": """
        SELECT
            payment_type,
            COUNT(*) AS trips,
            ROUND(
                100 * SUM(tip_amount)
                / NULLIF(SUM(total_amount), 0),
                2
            ) AS tip_percentage
        FROM hw3_taxi_clean
        WHERE total_amount > 0
        GROUP BY payment_type
        ORDER BY tip_percentage DESC
    """
}

with engine.connect() as conn:
    for question, sql in business_queries.items():
        print(f"\n{question}")
        result = conn.execute(text(sql))
        for row in result.mappings():
            print(dict(row))


Q1. Найзавантаженіші години
{'hour': 12, 'trips': 7708}
{'hour': 11, 'trips': 7006}
{'hour': 0, 'trips': 6461}
{'hour': 13, 'trips': 6327}
{'hour': 1, 'trips': 6160}

Q2. Середня вартість за перевізником
{'vendor_id': 2, 'trips': 77128, 'avg_amount': Decimal('30.53')}
{'vendor_id': 1, 'trips': 22871, 'avg_amount': Decimal('28.41')}

Q3. Чайові за способом оплати
{'payment_type': 1, 'trips': 75747, 'avg_tip': Decimal('4.71'), 'avg_total': Decimal('32.46')}
{'payment_type': 4, 'trips': 2129, 'avg_tip': Decimal('0.06'), 'avg_total': Decimal('2.02')}
{'payment_type': 3, 'trips': 846, 'avg_tip': Decimal('0.00'), 'avg_total': Decimal('9.04')}
{'payment_type': 2, 'trips': 21277, 'avg_tip': Decimal('0.00'), 'avg_total': Decimal('25.09')}

Q4. Найпопулярніші зони посадки
{'pu_location_id': 132, 'trips': 8035}
{'pu_location_id': 186, 'trips': 4302}
{'pu_location_id': 161, 'trips': 4073}
{'pu_location_id': 138, 'trips': 3879}
{'pu_location_id': 237, 'trips': 3360}

Q5. Середня дистанція за кільк

## SQL Analysis: Business Questions and Findings

### Q1. What are the busiest pickup hours?
**Business question:** At what hours is taxi demand highest?

**Finding:** The highest number of pickups occurred at 12:00 (7,708 trips), followed by 11:00 (7,006 trips). These results may help identify periods of increased demand. Hours are calculated using New York local time.

### Q2. Which vendor has a higher average trip amount?
**Business question:** How does average trip revenue differ between vendors?

**Finding:** Vendor 2 has a higher average trip amount ($30.53) compared with Vendor 1 ($28.41). However, this difference does not necessarily indicate better performance because trip characteristics may vary.

### Q3. How do tips differ by payment method?
**Business question:** Which payment methods are associated with higher recorded tips?

**Finding:** Payment type 1 has the highest average recorded tip ($4.71). Other payment types have nearly zero recorded tips. Cash tips may not be included in the dataset.

### Q4. Which pickup locations are the most popular?
**Business question:** Which pickup zones generate the highest taxi demand?

**Finding:** Location 132 has the highest pickup volume (8,035 trips), followed by locations 186 (4,302) and 161 (4,073). These zones may be important for taxi fleet allocation.

### Q5. Does trip distance vary by passenger count?
**Business question:** How does the number of passengers relate to average trip distance?

**Finding:** Trips with one passenger average 3.93 miles, while trips with four passengers average 5.26 miles. However, the relationship is not consistent across all passenger counts.

### Q6. What are the most expensive trips?
**Business question:** Which trips have the highest total amounts, and are there unusual records?

**Finding:** The most expensive trip costs $1,617.50 and covers 233.25 miles. Other unusual records include a $821 trip covering only 0.21 miles and a $500 trip with zero recorded distance. These observations require further investigation.

### Q7. What is the average trip duration?
**Business question:** How long does a taxi trip typically take?

**Finding:** The average trip duration is 16.45 minutes across 99,999 cleaned records. The mean may be influenced by unusually long trips.

### Q8. How does trip volume change by date?
**Business question:** Which dates have the highest recorded trip volumes?

**Finding:** January 1, 2024 has 70,495 recorded trips, compared with 29,490 on January 2. The dataset also contains 13 trips with pickup dates outside January 2024. Because only the first 100,000 records were selected, this distribution does not represent the entire month.

### Q9. Which pickup-to-dropoff routes are most frequent?
**Business question:** Which location pairs are most commonly used?

**Finding:** Route 132 → 132 is the most frequent, with 413 trips and an average distance of 1.29 miles. Route 237 → 236 follows with 391 trips. Popular routes may help identify recurring travel patterns.

### Q10. How does the recorded tip share vary by payment method?
**Business question:** What percentage of the total trip amount consists of recorded tips for each payment type?

**Finding:** Payment type 1 has a recorded tip share of 14.50%, compared with nearly zero for other payment types. The calculation includes only trips with positive total amounts and uses the total amount, including tips, as the denominator.

### Overall Conclusion

SQL analysis identified differences in trip demand, vendor revenue, passenger behavior, payment methods, and popular routes. Several potential outliers were also detected. These findings demonstrate how SQL aggregation, grouping, filtering, and sorting can support exploratory data analysis and business decision-making.

### Exploratory Data Analysis: Key Findings

The cleaned dataset contains 99,999 taxi trips.

The average trip distance is 4.20 miles, while the median is 2.00 miles. This difference suggests a right-skewed distribution, with a relatively small number of long trips increasing the average.

The average total amount is $30.04, compared with a median of $20.30. Extreme values include a maximum distance of 971.80 miles and a minimum total amount of -$426.54. These observations require further investigation.

Vendor 2 accounts for approximately 77% of trips, while payment type 1 represents approximately 76% of transactions. Single-passenger trips dominate the dataset, accounting for about 69% of records.

The dataset also contains zero-distance trips, zero-passenger records, and negative total amounts. These observations were retained for analysis because they may represent legitimate business cases or data quality issues.

**Limitation:** The dataset contains the first 100,000 records from the January 2024 Parquet file, rather than a random sample of the entire month. Therefore, the findings should not be generalized to all January taxi trips.

### Reflection

In this assignment, I worked with the NYC Yellow Taxi dataset and built a data processing pipeline using Python, Pandas, and PostgreSQL. I downloaded the Parquet dataset, selected 100,000 records, exported them to CSV, and loaded them into a staging table using PostgreSQL COPY.

The staging table stored all values as text, allowing me to separate raw data ingestion from validation and type conversion. I then created a clean table with appropriate data types, a primary key, NOT NULL constraints, and CHECK constraints. During transformation, one record was rejected because its drop-off time was earlier than its pickup time.

Data quality analysis revealed several interesting issues. Although no missing values were found in the four fields checked and no duplicate groups were detected, 1,047 trips had zero passengers, 1,811 had zero distance, and 1,586 had negative total amounts. Four trips exceeded 100 miles. These observations demonstrate why data validation is important before analysis.

Using SQL, I explored trip volumes, payment methods, passenger counts, popular routes, fares, tips, and trip durations. The average trip duration was 16.45 minutes, while the most common payment type accounted for approximately 76% of trips.

The main lesson was that a reproducible ETL pipeline requires more than loading data. It also requires explicit constraints, transparent cleaning rules, and careful interpretation of unusual records. I also learned the importance of testing the complete workflow in Google Colab.